# 🚀 SparseChron — 4D Interactive Viewer (August Model)

Streamlined, zero-bloat demo runner. Pre-configured for the **August High-Motion Model** (347k Gaussians, full jumping/leaping dynamic deformation).

### Requirements
- **Session options:** Accelerator = **GPU T4**, Internet = **ON**
- **Total setup time:** ~30–45 seconds (ZERO compilation, pre-cached CUDA kernels)

Run the 3 cells below in order.

In [ ]:
# 1. Clone repo + install dependencies (INSTANT ~15s, ZERO COMPILE)
import os, subprocess, sys

REPO = "/kaggle/working/SparseChron"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/rajhodedara/SparseChron.git", REPO], check=True)
else:
    print("Repo already cloned, skipping.")

%cd /kaggle/working/SparseChron

# Install gsplat wheel and requirements
WHEEL = "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/gsplat-1.5.3-py3-none-any.whl"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", WHEEL])
!pip install -q -r requirements.txt

# Unpack precompiled CUDA kernels directly into PyTorch JIT cache
print("Fetching precompiled gsplat CUDA kernels (0s compile time)...")
!mkdir -p /root/.cache/torch_extensions
!curl -sL https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/gsplat_cache.tar.gz | tar -xz -C /root/.cache/torch_extensions

# Touch cache timestamps to future so Ninja never rebuilds
for root, dirs, files in os.walk('/root/.cache/torch_extensions'):
    for f in files:
        try:
            os.utime(os.path.join(root, f), (2000000000, 2000000000))
        except Exception:
            pass

# Instant load precompiled C++ extension directly
from torch.utils.cpp_extension import _get_build_directory, _import_module_from_library
build_dir = _get_build_directory("gsplat_cuda", verbose=False)
_C = _import_module_from_library("gsplat_cuda", build_dir, is_python_module=True)

import gsplat.cuda._backend as _b
_b._C = _C
sys.modules["gsplat.csrc"] = _C

print("\n🚀 Environment ready! gsplat CUDA kernels hot and loaded.")

In [ ]:
# 2. Download August Model (347k Gaussians, dramatic motion) + Stage Code
import os, urllib.request

os.makedirs("/kaggle/working/outputs_mutant", exist_ok=True)
CKPT_PATH = "/kaggle/working/outputs_mutant/checkpoint_august_30000.ckpt"

if not os.path.exists(CKPT_PATH):
    print("Downloading August model checkpoint (237 MB)...")
    urllib.request.urlretrieve(
        "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/checkpoint_august_30000.ckpt",
        CKPT_PATH
    )
    print(f"Downloaded August model ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)!")
else:
    print("August checkpoint already cached.")

# Checkout August commit so shaders, deformation, and color math match exactly
!cd /kaggle/working/SparseChron && git checkout ce7fb83

print("\n✅ August model & renderer staged successfully!")

In [ ]:
# 3. Launch 4D Viewer (Viser) + Public Tunnel
import glob, subprocess, time

# Clean up any lingering viewer or tunnel processes
!pkill -9 -f interactive_viewer
!pkill -9 -f localtunnel
time.sleep(2)

# Grab Kaggle public IP for tunnel password
!curl -s ifconfig.me; echo ""

CKPT_PATH = "/kaggle/working/outputs_mutant/checkpoint_august_30000.ckpt"
print(f"Booting viewer with model: {CKPT_PATH}")

# Launch viewer and capture output
viewer_proc = subprocess.Popen(
    ["python", "/kaggle/working/SparseChron/scripts/interactive_viewer.py", "--checkpoint", CKPT_PATH, "--port", "8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Stream output until server confirms it is alive
print("Loading 347,000 Gaussians into GPU VRAM...")
for line in viewer_proc.stdout:
    print("  ", line.strip())
    if "Viser server running on port 8080" in line:
        break

print("\n✅ 4D Viewer LIVE on port 8080! Opening public tunnel...")
!npx -y localtunnel --port 8080